In [1]:
# =====================================================
# HIGH ACCURACY (90%+) WITH LR = 0.01 AND 5 EPOCHS
# =====================================================

!pip install kagglehub --quiet

import kagglehub
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import zipfile
import os

# -------------------------------
# DOWNLOAD DATASET
# -------------------------------
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset Path:", path)

if path.endswith(".zip"):
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall("/content/brain_tumor")
    dataset_path = "/content/brain_tumor"
else:
    dataset_path = path

train_dir = dataset_path + "/Training"
test_dir = dataset_path + "/Testing"

# -------------------------------
# STRONG DATA AUGMENTATION
# -------------------------------
img_size = 224
batch_size = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical"
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical"
)

# -------------------------------
# BUILD INCEPTIONV3 MODEL
# -------------------------------
base_model = InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(img_size, img_size, 3)
)

# Freeze entire model first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze LAST 20 layers only — very important for LR=0.01
for layer in base_model.layers[-20:]:
    layer.trainable = True

# -------------------------------
# CUSTOM CLASSIFIER
# -------------------------------
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)
x = Dropout(0.20)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)

output = Dense(4, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# -------------------------------
# COMPILE (LR = 0.01 as requested)
# -------------------------------
model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# -------------------------------
# TRAIN FOR 5 EPOCHS ONLY
# -------------------------------
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5
)

# -------------------------------
# EVALUATE
# -------------------------------
loss, acc = model.evaluate(test_data)
print(f"\n🔥 FINAL TEST ACCURACY: {acc * 100:.2f}%")


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset Path: /kaggle/input/brain-tumor-mri-dataset
Found 5712 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 960s 5s/step - accuracy: 0.7454 - loss: 0.8436 - val_accuracy: 0.8482 - val_loss: 0.3890
Epoch 2/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 941s 5s/step - accuracy: 0.8928 - loss: 0.3134 - val_accuracy: 0.9008 - val_loss: 0.2516
Epoch 3/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 953s 5s/step - accuracy: 0.9068 - loss: 0.2522 - val_accuracy: 0.9069 - val_loss: 0.2423
Epoch 4/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 977s 5s/step - accuracy: 0.9077 - loss: 0.2337 - val_accuracy: 0.9252 - val_loss: 0.1767
Epoch 5/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 982s 5s/step - accuracy: 0.9211 - loss: 0.2115 - val_accuracy: 0.9222 - val_loss: 0.1921
41/41 ━━━━━━━━━━━━━━━━━━━━ 166s 4s/step - accuracy: 0.9207 - loss: 0.1932

🔥 FINAL TEST ACCURACY: 92.22%
